In [3]:
import sys
import os
from pathlib import Path

sys.path.append(str(Path.cwd().parents[0]))

%load_ext autoreload
%autoreload 2

In [ ]:
# import tensorflow as tf

# devices = tf.config.list_physical_devices('GPU')
# print("Num GPUs Available:", len(devices))
# print(devices)

# for device in devices:
#     tf.config.experimental.set_memory_growth(device,True)



In [4]:
from sklearn.model_selection import StratifiedGroupKFold
from dataset.dataset import create_dataset, createOutputLabels


from constants import RANDOM_STATE

In [5]:
X, y, groups, metadata = create_dataset()

In [8]:
target_tensors = createOutputLabels(y)
y_stable = y["stable"].to_numpy()

In [9]:
import numpy as np
np.stack(target_tensors[:, 1])

array([0, 0, 0, ..., 0, 0, 0], shape=(5593,))

In [12]:
# -------------------------
# 80% train / 20% temp
# -------------------------
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    sgkf.split(X, y_stable, groups)
)

X_train = X.iloc[train_idx]
y_stable_train = y_stable[train_idx]
y_train = target_tensors[train_idx]
groups_train = groups[train_idx]

X_test = X.iloc[test_idx]
y_stable_test = y_stable[test_idx]
y_test = target_tensors[test_idx]
groups_test = groups[test_idx]


In [13]:
X_train.shape, X_test.shape

((4476, 26), (1117, 26))

In [14]:
from cv import cross_validation

results = cross_validation(
    X_train,
    y_train,
    y_stable_train,
    groups_train,
    ("mri", "pet", "cog", "csf", "rf"),
    run_name="proposed",
    n_splits=5,
    random_state=RANDOM_STATE,
    imputer="median",
    scaling="min-max",
    epochs=250,
    batch_size=32,
    time_weights=[0.75, 1, 1.5, 1.25],
    severity_matrix=[
        [0.0, 0.5, 2.0],
        [0.5, 0.0, 1.0],
        [2.0, 1.0, 0.0]
    ],
    severity_weight=0.5,
    transition_weight=1,
    transition_loss="huber",
    huber_delta=1.0,
    from_logits=False
)


===== Fold 1/5 =====

Fold 1: best_epoch=45, epochs_trained=60, best_val_loss=0.45492

===== Fold 2/5 =====
Fold 2: best_epoch=28, epochs_trained=43, best_val_loss=0.44762

===== Fold 3/5 =====
Fold 3: best_epoch=16, epochs_trained=31, best_val_loss=0.42211

===== Fold 4/5 =====
Fold 4: best_epoch=33, epochs_trained=48, best_val_loss=0.38014

===== Fold 5/5 =====
Fold 5: best_epoch=26, epochs_trained=41, best_val_loss=0.40939


In [3]:
from alz_prog_net.model import AlzProgNet
model = AlzProgNet(
    num_modalities=5,
    modalities_hidden_dims=[32, 256],
    modality_output_dim=128,
    latent_dim=512,
    num_transformer_layers=3,
    num_heads=2,
    ff_dim=512,
    dropout=0.1,
    progression_hidden_dims=(264, 128, 32),
    time_points=(0, 6, 12, 24),
    temporal_levels=(True, True, False),
    time_dim=16,
    n_time_frequencies=4,
    use_time=True,
    use_gate=True,
    use_residual=True,
    initial_residual_scale=0.1,
    output_dim=3
)

I0000 00:00:1789933012.793666 2382202 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 78752 MB memory:  -> device: 0, name: NVIDIA H100 80GB HBM3, pci bus id: 0000:d1:00.0, compute capability: 9.0a


In [4]:
model.summary(expand_nested=True)

Model: "alz_prog_net"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ modality_encoder                │ ?                      │   0 (unbuilt) │
│ (ModalityEncoder)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ modality_encoder_0_dense_0 │ ?                      │   0 (unbuilt) │
│ (Dense)                         │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └                            │ ?                      │             0 │
│ modality_encoder_0_dropout_0    │                        │               │
│ (Dropout)                       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ modality_encoder_0_dense_1 │ ?                      │   0 (unbuilt) │
│ (Dense)                         │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ modality_encoder_0_output  │ ?                      │   0 (unbuilt) │
│ (Dense)                         │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ modality_encoder_0_reshape │ ?                      │             0 │
│ (Reshape)                       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ modality_encoder_1              │ ?                      │   0 (unbuilt) │
│ (ModalityEncoder)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ modality_encoder_1_dense_0 │ ?                      │   0 (unbuilt) │
│ (Dense)                         │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └                            │ ?                      │             0 │
│ modality_encoder_1_dropout_0    │                        │               │
│ (Dropout)                       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ modality_encoder_1_dense_1 │ ?                      │   0 (unbuilt) │
│ (Dense)                         │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ modality_encoder_1_output  │ ?                      │   0 (unbuilt) │
│ (Dense)                         │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ modality_encoder_1_reshape │ ?                      │             0 │
│ (Reshape)                       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ modality_encoder_2              │ ?                      │   0 (unbuilt) │
│ (ModalityEncoder)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ modality_encoder_2_dense_0 │ ?                      │   0 (unbuilt) │
│ (Dense)                         │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └                            │ ?                      │             0 │
│ modality_encoder_2_dropout_0    │                        │               │
│ (Dropout)                       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ modality_encoder_2_dense_1 │ ?                      │   0 (unbuilt

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [17]:
# from keras.utils import plot_model

# # Save the model architecture as a PNG file
# plot_model(alz_prog_net, to_file='alz_prog_net_summary.png', show_shapes=True, show_layer_names=True, expand_nested=True)

In [18]:
# LongitudinalTransitionLoss(
#         class_weights=class_weights,
#         time_weights=[0.75, 1, 1.5, 1.25],
#         severity_matrix=[
#             [0.0, 0.5, 2.0],
#             [0.5, 0.0, 1.0],
#             [2.0, 1.0, 0.0]
#         ],
#         severity_weight=0.5,
#         transition_weight=1,
#         transition_loss="huber",
#         huber_delta=1.0,
#         from_logits=False
#     )